In [ ]:
import pandas as pd
import numpy as np
import re
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings('ignore')

# ==================== CONFIGURATION ====================
CONFIG = {
    'train_csv': 'student_resource/dataset/train.csv',
    'test_csv': 'student_resource/dataset/test.csv',
    'output_path': 'student_resource/dataset/test_out.csv',
    'use_sample': False,
    'sample_size': 10000,
    'n_folds': 5,
    'random_state': 42,
}

# ==================== SMAPE METRIC ====================
def calculate_smape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    predicted = np.maximum(predicted, 0.01)
    numerator = np.abs(predicted - actual)
    denominator = (np.abs(actual) + np.abs(predicted)) / 2
    smape = np.mean(numerator / (denominator + 1e-10)) * 100
    return smape

# ==================== FEATURE EXTRACTION ====================
def extract_value_field(text):
    if pd.isna(text):
        return np.nan
    match = re.search(r'Value:\s*(\d+\.?\d*)', str(text))
    return float(match.group(1)) if match else np.nan

def extract_pack_info_comprehensive(text):
    if pd.isna(text):
        return 1, 'none'
    text_str = str(text).lower()
    value_match = re.search(r'value:\s*(\d+\.?\d*)', text_str)
    if value_match:
        val = float(value_match.group(1))
        if val > 1:
            return val, 'value_field'
    pack_patterns = [
        r'\(pack of (\d+)\)', r'pack of (\d+)', r'\((\d+) pack\)', r'(\d+) pack', r'(\d+)-pack'
    ]
    for pattern in pack_patterns:
        match = re.search(pattern, text_str)
        if match:
            return float(match.group(1)), 'pack'
    case_match = re.search(r'(\d+)\s*per case|case of (\d+)', text_str)
    if case_match:
        for group in case_match.groups():
            if group:
                return float(group), 'case'
    count_match = re.search(r'(\d+)\s*count', text_str)
    if count_match:
        count = float(count_match.group(1))
        if count > 1:
            return count, 'count'
    return 1, 'single'

def extract_unit_info(text):
    if pd.isna(text):
        return 0, 'none', 0
    text_str = str(text).lower()
    patterns = [
        (r'(\d+\.?\d*)\s*ounce', 'ounce', 1),
        (r'(\d+\.?\d*)\s*oz', 'ounce', 1),
        (r'(\d+\.?\d*)\s*fl oz', 'fl_oz', 1),
        (r'(\d+\.?\d*)\s*pound', 'pound', 16),
        (r'(\d+\.?\d*)\s*lb', 'pound', 16),
        (r'(\d+\.?\d*)\s*ml', 'ml', 0.033814),
        (r'(\d+\.?\d*)\s*liter', 'liter', 33.814),
        (r'(\d+\.?\d*)\s*gram', 'gram', 0.035274),
        (r'(\d+\.?\d*)\s*kg', 'kg', 35.274),
    ]
    for pattern, unit, conversion in patterns:
        match = re.search(pattern, text_str)
        if match:
            value = float(match.group(1))
            return value, unit, value * conversion
    return 0, 'none', 0

# ==================== ADVANCED FEATURES ====================
def create_advanced_features(df):
    """Extract comprehensive features from catalog content"""
    features = pd.DataFrame()
    text = df['catalog_content'].fillna('')
    text_lower = text.str.lower()
    
    # === PACK AND UNIT FEATURES ===
    pack_info = text.apply(extract_pack_info_comprehensive)
    features['pack_quantity'] = pack_info.apply(lambda x: x[0])
    pack_type_map = {'none': 0, 'single': 1, 'value_field': 2, 'pack': 3, 'case': 4, 'count': 5}
    features['pack_type_encoded'] = pack_info.apply(lambda x: x[1]).map(pack_type_map).fillna(0)
    features['is_multi_pack'] = (features['pack_quantity'] > 1).astype(int)
    features['log_pack_qty'] = np.log1p(features['pack_quantity'])
    
    unit_info = text.apply(extract_unit_info)
    features['unit_size'] = unit_info.apply(lambda x: x[0])
    features['unit_oz_equiv'] = unit_info.apply(lambda x: x[2])
    features['has_unit_size'] = (features['unit_size'] > 0).astype(int)
    features['total_volume'] = features['pack_quantity'] * features['unit_oz_equiv']
    features['log_total_volume'] = np.log1p(features['total_volume'])
    features['volume_per_pack'] = features['total_volume'] / (features['pack_quantity'] + 1)
    
    # === VALUE FIELD ===
    features['value_field'] = text.apply(extract_value_field)
    features['has_value_field'] = (~features['value_field'].isna()).astype(int)
    features['value_field_filled'] = features['value_field'].fillna(1)
    features['value_to_pack_ratio'] = features['value_field_filled'] / (features['pack_quantity'] + 1)
    
    # === BRAND INDICATORS ===
    premium_brands = ['organic', 'premium', 'gourmet', 'artisan', 'handcrafted', 'imported', 'specialty', 'luxury']
    budget_brands = ['value', 'basic', 'economy', 'budget', 'generic']
    features['is_premium'] = text_lower.apply(lambda x: any(b in x for b in premium_brands)).astype(int)
    features['is_budget'] = text_lower.apply(lambda x: any(b in x for b in budget_brands)).astype(int)
    
    # === TEXT STRUCTURE ===
    features['text_length'] = text.str.len()
    features['word_count'] = text.str.split().str.len()
    features['bullet_count'] = text.str.count('Bullet Point')
    features['has_bullets'] = (features['bullet_count'] > 0).astype(int)
    
    # Extract title
    titles = text.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    features['title_length'] = titles.str.len()
    features['title_to_total_ratio'] = features['title_length'] / (features['text_length'] + 1)
    
    # === NUMBER FEATURES ===
    all_numbers = text.apply(lambda x: [float(n) for n in re.findall(r'\d+\.?\d*', str(x))] if pd.notna(x) else [])
    features['num_count'] = all_numbers.apply(len)
    features['num_density'] = features['num_count'] / (features['text_length'] + 1) * 1000
    features['max_number'] = all_numbers.apply(lambda x: max(x) if x else 0)
    features['sum_numbers'] = all_numbers.apply(lambda x: sum(x) if x else 0)
    features['has_small_nums'] = all_numbers.apply(lambda x: any(0.1 <= n <= 10 for n in x)).astype(int)
    features['has_medium_nums'] = all_numbers.apply(lambda x: any(10 < n <= 100 for n in x)).astype(int)
    features['has_large_nums'] = all_numbers.apply(lambda x: any(n > 100 for n in x)).astype(int)
    
    # === KEYWORD FEATURES ===
    keywords = {
        'bulk': ['bulk', 'wholesale', 'case', 'carton'],
        'individual': ['single', 'individual', 'piece', 'unit'],
        'family': ['family size', 'family pack', 'party size'],
        'trial': ['sample', 'trial', 'mini', 'travel'],
    }
    for category, words in keywords.items():
        features[f'is_{category}'] = text_lower.apply(lambda x: any(w in x for w in words)).astype(int)
    
    # === CATEGORY HINTS ===
    categories = {
        'food': ['food', 'edible', 'snack', 'beverage', 'drink'],
        'beauty': ['beauty', 'cosmetic', 'skincare', 'makeup'],
        'household': ['cleaning', 'detergent', 'paper', 'towel'],
    }
    for cat, words in categories.items():
        features[f'cat_{cat}'] = text_lower.apply(lambda x: any(w in x for w in words)).astype(int)
    
    # === INTERACTIONS ===
    features['volume_x_premium'] = features['log_total_volume'] * features['is_premium']
    features['pack_x_bulk'] = features['log_pack_qty'] * features['is_bulk']
    
    return features.fillna(0)

# ==================== TARGET ENCODING ====================
def add_target_encoded_features(train_df, test_df, target_col='price'):
    """Add target-encoded categorical features"""
    
    # IMPORTANT: Reset index to avoid KeyError with sampled data
    train_df = train_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    # Extract categorical features
    train_df['first_word'] = train_df['catalog_content'].str.split().str[0].fillna('UNKNOWN')
    test_df['first_word'] = test_df['catalog_content'].str.split().str[0].fillna('UNKNOWN')
    
    train_df['potential_brand'] = train_df['catalog_content'].str.extract(
        r'Item Name:\s*([A-Z][a-z]+)', expand=False
    ).fillna('UNKNOWN')
    test_df['potential_brand'] = test_df['catalog_content'].str.extract(
        r'Item Name:\s*([A-Z][a-z]+)', expand=False
    ).fillna('UNKNOWN')
    
    categorical_features = ['first_word', 'potential_brand']
    
    # K-Fold target encoding
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    for cat_col in categorical_features:
        train_df[f'{cat_col}_target_enc'] = 0.0
        global_mean = train_df[target_col].mean()
        
        # OOF encoding for train
        for train_idx, val_idx in kf.split(train_df):
            means = train_df.iloc[train_idx].groupby(cat_col)[target_col].mean()
            train_df.iloc[val_idx, train_df.columns.get_loc(f'{cat_col}_target_enc')] = \
                train_df.iloc[val_idx][cat_col].map(means).fillna(global_mean).values
        
        # For test, use all training data
        category_means = train_df.groupby(cat_col)[target_col].mean()
        test_df[f'{cat_col}_target_enc'] = test_df[cat_col].map(category_means).fillna(global_mean)
        
        # Smoothed version
        category_counts = train_df.groupby(cat_col).size()
        smooth_means = (category_means * category_counts + global_mean) / (category_counts + 1)
        train_df[f'{cat_col}_smooth_enc'] = train_df[cat_col].map(smooth_means).fillna(global_mean)
        test_df[f'{cat_col}_smooth_enc'] = test_df[cat_col].map(smooth_means).fillna(global_mean)
    
    # Drop original categorical columns
    train_df = train_df.drop(categorical_features, axis=1)
    test_df = test_df.drop(categorical_features, axis=1)
    
    return train_df, test_df

# ==================== TFIDF FEATURES ====================
def create_multiple_tfidf_features(train_texts, test_texts):
    """Create multiple TF-IDF representations"""
    
    # Full text TF-IDF
    tfidf_full = TfidfVectorizer(
        max_features=3000,
        ngram_range=(1, 4),
        min_df=2,
        max_df=0.9,
        sublinear_tf=True,
        stop_words='english'
    )
    train_tfidf_full = tfidf_full.fit_transform(train_texts).toarray()
    test_tfidf_full = tfidf_full.transform(test_texts).toarray()
    
    # Title-only TF-IDF
    train_titles = train_texts.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    test_titles = test_texts.str.extract(r'Item Name:([^\n]+)', expand=False).fillna('')
    
    tfidf_title = TfidfVectorizer(
        max_features=1000,
        ngram_range=(1, 3),
        min_df=2,
        sublinear_tf=True,
        stop_words='english'
    )
    train_tfidf_title = tfidf_title.fit_transform(train_titles).toarray()
    test_tfidf_title = tfidf_title.transform(test_titles).toarray()
    
    # Character n-grams
    tfidf_char = TfidfVectorizer(
        max_features=500,
        analyzer='char',
        ngram_range=(3, 5),
        min_df=5,
        sublinear_tf=True
    )
    train_tfidf_char = tfidf_char.fit_transform(train_titles).toarray()
    test_tfidf_char = tfidf_char.transform(test_titles).toarray()
    
    train_combined = np.hstack([train_tfidf_full, train_tfidf_title, train_tfidf_char])
    test_combined = np.hstack([test_tfidf_full, test_tfidf_title, test_tfidf_char])
    
    print(f"✓ TF-IDF features: {train_combined.shape[1]}")
    return train_combined, test_combined

# ==================== MODEL PARAMETERS ====================
lgbm_params = {
    'objective': 'regression',
    'metric': 'mae',
    'n_estimators': 3000,
    'learning_rate': 0.01,
    'max_depth': 10,
    'num_leaves': 80,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

xgb_params = {
    'objective': 'reg:squarederror',
    'n_estimators': 3000,
    'learning_rate': 0.01,
    'max_depth': 9,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 2.0,
    'gamma': 0.1,
    'random_state': 43,
    'tree_method': 'hist',
    'n_jobs': -1,
    'early_stopping_rounds': 200  # Moved here from .fit()
}

catboost_params = {
    'iterations': 3000,
    'learning_rate': 0.01,
    'depth': 8,
    'l2_leaf_reg': 3.0,
    'random_seed': 44,
    'loss_function': 'MAE',
    'verbose': False
}

# ==================== STACKING ENSEMBLE ====================
def train_stacking_ensemble(X_train, y_train, X_test, n_folds=5):
    """Train stacking ensemble with meta-model"""
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
    oof_lgbm = np.zeros(len(X_train))
    oof_xgb = np.zeros(len(X_train))
    oof_cat = np.zeros(len(X_train))
    
    test_lgbm = np.zeros(len(X_test))
    test_xgb = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))
    
    fold_smapes = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train), 1):
        print(f"\n{'='*70}\n📁 FOLD {fold}/{n_folds}\n{'='*70}")
        
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        
        # LightGBM
        print("Training LightGBM...")
        lgbm = lgb.LGBMRegressor(**lgbm_params)
        lgbm.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                 callbacks=[lgb.early_stopping(200, verbose=False)])
        oof_lgbm[val_idx] = lgbm.predict(X_val)
        test_lgbm += lgbm.predict(X_test) / n_folds
        lgbm_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_lgbm[val_idx]))
        print(f"  SMAPE: {lgbm_smape:.2f}%")
        
        # XGBoost
        print("Training XGBoost...")
        xgb_model = xgb.XGBRegressor(**xgb_params)
        xgb_model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx] = xgb_model.predict(X_val)
        test_xgb += xgb_model.predict(X_test) / n_folds
        xgb_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_xgb[val_idx]))
        print(f"  SMAPE: {xgb_smape:.2f}%")
        
        # CatBoost
        print("Training CatBoost...")
        cat_model = CatBoostRegressor(**catboost_params)
        cat_model.fit(X_tr, y_tr, eval_set=(X_val, y_val), 
                      early_stopping_rounds=200, verbose=False)
        oof_cat[val_idx] = cat_model.predict(X_val)
        test_cat += cat_model.predict(X_test) / n_folds
        cat_smape = calculate_smape(np.expm1(y_val), np.expm1(oof_cat[val_idx]))
        print(f"  SMAPE: {cat_smape:.2f}%")
        
        # Simple average for this fold
        fold_pred = (oof_lgbm[val_idx] + oof_xgb[val_idx] + oof_cat[val_idx]) / 3
        fold_smape = calculate_smape(np.expm1(y_val), np.expm1(fold_pred))
        fold_smapes.append(fold_smape)
        print(f"\n🎯 Fold {fold} Average SMAPE: {fold_smape:.2f}%")
    
    # Meta-model
    print("\n" + "="*70)
    print("Training Meta-Model...")
    print("="*70)
    meta_features = np.column_stack([oof_lgbm, oof_xgb, oof_cat])
    meta_model = Ridge(alpha=1.0, random_state=42)
    meta_model.fit(meta_features, y_train)
    
    test_meta_features = np.column_stack([test_lgbm, test_xgb, test_cat])
    test_preds_log = meta_model.predict(test_meta_features)
    
    oof_preds_log = meta_model.predict(meta_features)
    oof_smape = calculate_smape(np.expm1(y_train), np.expm1(oof_preds_log))
    
    print(f"\n📊 Cross-Validation Results:")
    print(f"   Fold SMAPEs: {[f'{s:.2f}%' for s in fold_smapes]}")
    print(f"   Mean: {np.mean(fold_smapes):.2f}% ± {np.std(fold_smapes):.2f}%")
    print(f"\n🎯 Meta-Model OOF SMAPE: {oof_smape:.2f}%")
    print(f"\n📊 Meta-Model Weights:")
    print(f"   LGBM:     {meta_model.coef_[0]:.4f}")
    print(f"   XGB:      {meta_model.coef_[1]:.4f}")
    print(f"   CatBoost: {meta_model.coef_[2]:.4f}")
    
    return np.expm1(test_preds_log), oof_smape

# ==================== MAIN PIPELINE ====================
def main():
    print("="*80)
    print("🚀 ADVANCED MODEL - TARGET: 45% SMAPE")
    print("="*80)
    
    # Load data
    print("\n📂 Loading data...")
    train_df = pd.read_csv(CONFIG['train_csv'])
    test_df = pd.read_csv(CONFIG['test_csv'])
    
    if CONFIG['use_sample']:
        train_df = train_df.sample(n=min(CONFIG['sample_size'], len(train_df)), random_state=42).reset_index(drop=True)
        test_df = test_df.sample(n=min(CONFIG['sample_size']//5, len(test_df)), random_state=42).reset_index(drop=True)
        print(f"⚠️  Using sample: {len(train_df)} train, {len(test_df)} test")
    else:
        train_df = train_df.reset_index(drop=True)
        test_df = test_df.reset_index(drop=True)
        print(f"✓ Full dataset: {len(train_df):,} train, {len(test_df):,} test")
    
    train_df['catalog_content'] = train_df['catalog_content'].fillna('')
    test_df['catalog_content'] = test_df['catalog_content'].fillna('')
    
    # Step 1: Advanced features
    print("\n🔧 Creating advanced features...")
    train_text_feat = create_advanced_features(train_df)
    test_text_feat = create_advanced_features(test_df)
    print(f"✓ Text features: {train_text_feat.shape[1]}")
    
    # Step 2: Target encoding
    print("\n🎯 Adding target-encoded features...")
    train_df_enc, test_df_enc = add_target_encoded_features(
        train_df.copy(), test_df.copy(), target_col='price'
    )
    target_enc_cols = [col for col in train_df_enc.columns if '_enc' in col]
    train_target_enc = train_df_enc[target_enc_cols].values
    test_target_enc = test_df_enc[target_enc_cols].values
    print(f"✓ Target-encoded features: {len(target_enc_cols)}")
    
    # Step 3: TF-IDF
    print("\n📝 Creating TF-IDF features...")
    train_tfidf, test_tfidf = create_multiple_tfidf_features(
        train_df['catalog_content'], test_df['catalog_content']
    )
    
    # Step 4: Combine all features
    print("\n🔗 Combining all features...")
    X_train = np.hstack([train_tfidf, train_text_feat.values, train_target_enc])
    X_test = np.hstack([test_tfidf, test_text_feat.values, test_target_enc])
    print(f"✓ Final feature count: {X_train.shape[1]}")
    
    y_train = np.log1p(train_df['price'].values)
    
    # Step 5: Train stacking ensemble
    print("\n" + "="*80)
    print("🏋️  TRAINING STACKING ENSEMBLE")
    print("="*80)
    predictions, oof_smape = train_stacking_ensemble(X_train, y_train, X_test, CONFIG['n_folds'])
    
    # Save submission
    predictions = np.maximum(predictions, 0.1)
    submission = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': predictions})
    submission.to_csv(CONFIG['output_path'], index=False)
    
    # Final results
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE!")
    print("="*80)
    print(f"\n🎯 Expected Leaderboard Score: ~{oof_smape:.1f}%")
    print(f"💾 Submission saved: {CONFIG['output_path']}")
    print(f"\n📈 Prediction Statistics:")
    print(f"   Count:  {len(predictions):,}")
    print(f"   Min:    ${predictions.min():.2f}")
    print(f"   Max:    ${predictions.max():.2f}")
    print(f"   Mean:   ${predictions.mean():.2f}")
    print(f"   Median: ${np.median(predictions):.2f}")
    
    print("\n🏆 COMPARISON:")
    print(f"   Baseline:    50.00% SMAPE")
    print(f"   Your model:  {oof_smape:.2f}% SMAPE")
    if oof_smape < 50:
        improvement = 50 - oof_smape
        print(f"   🎉 IMPROVEMENT: {improvement:.2f}% better!")
    print("="*80)

if __name__ == "__main__":
    main()